# 12 - Live demo: Attribute Routing Layer (Gradio)

Runs the submitted ARL (trained router + governance flags + flag-gated text on frozen SDXL) on your own sketch, reference and caption, next to the fixed baseline.

Needs a **T4 GPU** runtime and Drive access to `sketchy_dataset_one_sketch/governance/governance.pt`. Run the cells in order; the last cell prints a public `gradio.live` link.


In [ ]:
%%bash
pip uninstall -y torchao -q
pip install -q -U diffusers transformers accelerate safetensors peft lpips opencv-python-headless gradio
nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os, sys, glob, json, zipfile, base64, zlib, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")   # only needed to read governance.pt
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
GOV = f"{DRIVE_ROOT}/governance/governance.pt"
REPO = "/content/repo"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "governed-pipeline", "https://github.com/Harshitha-282007/IE643_final.git", REPO], check=True)
p = f"{REPO}/src/pipeline.py"
src = open(p).read()
if "text_flags" not in src:
    MIX = '''    @staticmethod
    def mix_text(pr, pooled, tn, pn, flags):
        """Flag-gated text conditioning: each attribute's neutral caption is blended in with weight = its flag."""
        for w, f in (("colour", float(flags[0])), ("material", float(flags[1]))):
            if tn.get(w) is not None and f:
                pr = pr - f * (pr - tn[w].to(pr.dtype))
                pooled = pooled - f * (pooled - pn[w].to(pooled.dtype))
        return pr, pooled

'''
    RUNBLK = '''        if text_flags is not None:
            nz = it.get("txt_neutral") or {}
            tn = {w: None if v is None else torch.cat([npe, v[0].to(dev)]) for w, v in nz.items()}
            pn = {w: None if v is None else torch.cat([nppe, v[1].to(dev)]) for w, v in nz.items()}
            pr, pooled = self.mix_text(pr, pooled, tn, pn, text_flags)
'''
    a = "    @torch.no_grad()\n    def run_role("
    b = "    def run(self, it, p=None, shaped=True, gates=None, split=True, style_only=None, guided=False, seed_off=0):"
    c = "        governed = gates is not None\n"
    assert src.count(a) == 1 and src.count(b) == 1 and src.count(c) == 1
    src = src.replace(a, MIX + a).replace(b, b[:-2] + ", text_flags=None):").replace(c, RUNBLK + c)
    open(p, "w").write(src)
open(f"{REPO}/src/pilot_flag_gate.py", "wb").write(zlib.decompress(base64.b64decode("eNqdWW2P47YR/u5fQagIVkp0uvWlbYJNheKu94Kgl+thE/SLsRBoi7Z5liWVpHfXue5/7zNDSpa83pd0cbElcjhvnHlmxomiaPJZV427EGWjrFhWcvViJZ0qhVO3TiyautRON7WuV2LbXCusVM3OiJdiCyqjZZUKWZf+tKYD27axStxotxZurYS61dbRadPscODvk8m/agXm1ol23bgmBZVRxLdeVnrhQHdjRdzMv6iFS4O4tJeWpGJJ8qXZ2lToUtVOL2QlrILKL4VRtql2pDBerFOtvZhMBP5eF0t9C5Lh32fdqkrXKjO7OrZr2aoyfy8rqxLx+J9ntdrpUtYLJebSMh8R/+Pdp9/eXSYs8c15YRdrVe6qg9hTEn8zuycFknecaaoL0bH0Dl7AfKPEtax2ynqphffzg3a2eSBopZHkw0fU6O+uUtLUMOO1c0bPcVxcggntfJR7ZVjy24KCh2LnGR5OObz4hM3jZYFrxsUWuOWElXgtvrsXi7GFQFi+WMt6BaOteJ0EyWtpxlf7XMnTVEyT5KSnmQLBZhakQCOmLOtd8O/Q2Gf7dyibhL45ZWU4TknF/F2zUkgkk0wm71klpJgSS9NsOb9WSEpTcxzWyt00ZiPibVOqygokwVujr1Xyk9BLSk5txa6W11JXcg5H0nFDsYRXopcIpZ2FIiTbb9Y4M+FtiuKzxsgF6NmAs0x8atyaggB8nZHwQJmJn7dypWB2q8wLTc/C6S2IsLRViJ6F7dlTyusVi70x2jlVQ70StsMDRanNTxMF2/YisFlL5yPSkkC70S38Cv82QiLvSVdk/26rDgKsAiDJWswJhsC+hIKTCJCnAVLGiS+2qbvnxnZP0FdNupd6t233QlpRt91SC/ZYwL+2nPA9fP75owibbH5/WiEzETCVYvJ3l558cGWBrrsAKz4ek2RLJd3OELp6WokcLNq1Aeh4WuJPkZK5ZqPqnq5WO1xKVSxkS3g4mby+/OVXkYtZFLAwSkU0BCl+D+FNz4eE9m+UZPR0LwWiq8k/f/70lpjHEYCbiJDQ9IWEjhC4k1ItRasXm4IxPybELAjokRGktP5dAeNtW2mXR1QbcJYQPZ8mF5x2uLT32qBmHAoH8ABmIip1mXg0HNcMjoKubAAi8VUjdG7WVJ6Owx6pQuS8OfEwcFTmBFKrtF04BedqSyH/nsvBXAErFNh0AYsgXalaGcrtjKKO+HJ1wyUYQhZhhK7Fx0zdUkyxQ4auQQoVbt+qfBq8QR+p2Oi6tDl7PKHENrOIPRddiTz3TrxiWfM9JH294+deGrH2PvUkGZIEtyN3lYvBqTF6pWuEjS6jqxSnk9m7y4xEFs0yNglkCNNzbHRJPP1FxPN9cuC8AuF8PwPFVb8GZevGUVrGQf2/5fy2GhzsQFjXO3V8Ml7NOMCuZhEFnuWHglc8YM448MbbvNJtU0COt3nlOfLH+RQPAvcgNmyS6yIfQlFCgfCpQXsAhz3OJCg3ZNKF34HNE5oaBbSo6WZSMVvNNld8URu6Jva5vw2gNbLgclcT2L0zpjFxVDfD7LLSabvUwB2PpBVSi9oqypw+pQG6RUvtY/w5FT5oS+kkgXfaoXhKdSknzdPQA4aXje6WqZnLCZ5SUTWrvDW6duFGHgBMrnCElx9+OZC1oQ53RJQuBaxnisZmW7lRUMjGvWZcTopm47sfT9cicCu5nZeoKlDywkN61rSqjpfR186+u5dfDbx7F/ljn7PWKJR8VXgUKljQzGQo3nFk1DKs4x5xHVF4GafllWdlW7VgjCjRDscbtc+Rlz4d7Ua5xTpv2hiqRf4tQksM/t0iHpVRcBWth0giBoOggvA8KBZWEY9OrRqzB+4ybXi7StLJ/Wa0xyJy7j3Q8OoEc/MTDki5hOWDOgZcSY5cwWK1U1vyBDvid93Cn94TR8Qp3E+XW/CBmB2YJInv/HW9bJ7EQTRHuRfXiTigFtcKMNhQT9hIF197ZTepuGZOPZxkXn5y159FUJQ4++EXCg8yo2DSGCmB6cXNols4zKMLt2cEcowVKEyKn3pW6MzAiRmC3n8fHyBVDzbB8N6azouE5PkI0R+KkuPQeH7IhDAdnBzun4qowR9FCiGnscPY8Ssh1IsOFIcUB6AMxZK8Sw9kC/kltPz0mLL7wgI9PqETExbUGudRuEfwffgaIt8nx4MuI4lGg8cMzPralIrwxvj/pIM8hvqvzE8ZsXa+FfCzw8lASnq2wNi4hxTqgbMSXa6NKcJHQTOqHnFEMcO9XQgSQpvesfR28Cq9HcxlSt9XJnddXlL9QH8UA/wpFfDfoAi7JS6NkFjPhnygLxYCq1FfAS7U/fS97bhKhqL4mSc0kjWa9k/xGTXGF6dv4k+DsVxiJutnc9SlN+l4OEe3SuWBkr+7Ja5Z3ZnkWfoeytSxul3f/oTdmE7bp7kNOv8/4MdRgLvlA5x5irh4LMAHbn17RuNuqTBzob/nJn+OmfP+bC5iVfPrFk5nQriDO5dSOVSdxiT/pyHxNDtPBT6SkwbdH4b+6A086Dbfo/2bwsd3aJDZ1bQwqcLiNS84dlAu0KCE5ubuZZi7M8pwP3xQs0f1lHMepayMuavhwwkDCLokesn8nB12PIY8XkBpmXyCDermxk44oRxNxi+/gpT7qDNdnl3dZW29ikYH72vECl1MTv0+NmqD77V8gQ1eauSq53OyAewhiJybLXalzOy+XqxNQ416nPwk3DmMIV9m9BGPjw0gbdhNpGNkTTIrr4MWz5VKlzeDD8lp/+2dFvFE1uzqMh6oJF5Ay1S8GjPvsZ5J0awfrh8IfUOlExCPDMLcPTpIFSP65sX0Ryu+efHqFT6z6dJG4hvhbe2ryYNKhuwJP/mEZsQomyMzSBP+rRYv/B0mAozXjvjbxA8X+YPtH/k2p49TtXOxRJ2FUSiaLHVhc87pdVOV+Xn2A463+P5LSpT5D8h06jIqeYvF773aB8d5A4LnhgHtN3yyHTszFWHCzq0zIYVNaPJDp7ukcWhYuMepdSLjnmpbebyX3DTWbSatNEbu4/EwM07HUDuPEjLJYBrmL/RZlx/ekFlghwFt+tdxjDyKAB60oMpj8k/BwbH0e1ztAlzhu8wuGqNiTdHoW+swHl35N/SI9DjuhA4ovbCZbKFVGXOM6PIwdJ1qmaFpzpHf6ltYsFwW17Z4nfv5gNw9t/HA66RV7zXkJt1Lkm2VrOMkeaLfo79vv6Xe7Ho0dthFN250v45oq2vraEyOrymjDDdbHQ/6ea4ts7eYX98bgkAyerCduaZY2OvRtbBPbYblyEfz7XHLFOoba+LzMBW/Th7a4YHG/3RAiC0xD9m1QgsfqPxPCP143v8uEBC/G/6la6vGVXp+ejVr9/TEv9BWzlctyll5Sy6oECq7ORHYuEIYkkzc6Cvxnag8dtMrDlgAbx6/yn4U34p4tI3977HYn6aR4z87pUA/cA9dlk596qp6t+VfAv2B5w2e8naG8+cYK7d23dzEw6BGQei3rXKF0w4DB+5uFKh3F4LSKTTsZ1eziz+fA43p/6HVju0bJLHnNx2J46zpZU3HsrBJ/HkGOztMaWfJXTpa7kYzbDwgmlz1Je3g4+As9vYYSbwmdB1fek3/MKgkx12Gpmbu/D5mjaQdbOfU7w358XDfkuHvNjOo7lU8UF3y6VunFxsbz9ilvLLvV5iUotPp1doVldzDECr8HLHgh5A8rjiHDGK7UGdanU+n58kwAx8/M/kfOnJV3Q==")))
sys.path.insert(0, f"{REPO}/src")
print("patched:", "text_flags" in open(p).read(), "| governance.pt:", os.path.exists(GOV))


In [ ]:
import base64, os
os.makedirs('/content/examples', exist_ok=True)
EX = {"sketch.jpg": "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAMCAgICAgMCAgIDAwMDBAYEBAQEBAgGBgUGCQgKCgkICQkKDA8MCgsOCwkJDRENDg8QEBEQCgwSExIQEw8QEBD/2wBDAQMDAwQDBAgEBAgQCwkLEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBD/wAARCAGkAaQDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwD4tT/goH+2ZGoRP2gfEoA6D9z/APG6d/w8G/bO/wCjgvEv/kH/AON189UUAfQv/Dwb9s7/AKOC8S/+Qf8A43R/w8G/bO/6OC8S/wDkH/43Xz1RQB9C/wDDwb9s7/o4LxL/AOQf/jdH/Dwb9s7/AKOC8S/+Qf8A43Xz1RQB9C/8PBv2zv8Ao4LxL/5B/wDjdH/Dwb9s7/o4LxL/AOQf/jdfPVFAH0L/AMPBv2zv+jgvEv8A5B/+N0f8PBv2zv8Ao4LxL/5B/wDjdfPVFAH0L/w8G/bO/wCjgvEv/kH/AON0f8PBv2zv+jgvEv8A5B/+N189UUAfQv8Aw8G/bO/6OC8S/wDkH/43R/w8G/bO/wCjgvEv/kH/AON189UUAfQv/Dwb9s7/AKOC8S/+Qf8A43R/w8G/bO/6OC8S/wDkH/43Xz1RQB9C/wDDwb9s7/o4LxL/AOQf/jdH/Dwb9s7/AKOC8S/+Qf8A43Xz1RQB9C/8PBv2zv8Ao4LxL/5B/wDjdH/Dwb9s7/o4LxL/AOQf/jdfPVFAH0L/AMPBv2zv+jgvEv8A5B/+N0f8PBv2zv8Ao4LxL/5B/wDjdfPVFAH0L/w8G/bO/wCjgvEv/kH/AON0f8PBv2zv+jgvEv8A5B/+N189UUAfQv8Aw8G/bO/6OC8S/wDkH/43R/w8G/bO/wCjgvEv/kH/AON189UUAfQv/Dwb9s7/AKOC8S/+Qf8A43R/w8G/bO/6OC8S/wDkH/43Xz1RQB9C/wDDwb9s7/o4LxL/AOQf/jdH/Dwb9s7/AKOC8S/+Qf8A43Xz1RQB9C/8PBv2zv8Ao4LxL/5B/wDjdH/Dwb9s7/o4LxL/AOQf/jdfPVFAH0L/AMPBv2zv+jgvEv8A5B/+N0f8PBv2zv8Ao4LxL/5B/wDjdfPVFAH0L/w8G/bO/wCjgvEv/kH/AON0f8PBv2zv+jgvEv8A5B/+N189UUAfQv8Aw8G/bO/6OC8S/wDkH/43R/w8G/bO/wCjgvEv/kH/AON189UUAfQv/Dwb9s7/AKOC8S/+Qf8A43R/w8G/bO/6OC8S/wDkH/43Xz1RQB9C/wDDwb9s7/o4LxL/AOQf/jdH/Dwb9s7/AKOC8S/+Qf8A43Xz1RQB9C/8PBv2zv8Ao4LxL/5B/wDjdH/Dwb9s7/o4LxL/AOQf/jdfPVFAH0L/AMPBv2zv+jgvEv8A5B/+N0f8PBv2zv8Ao4LxL/5B/wDjdfPVFAH0L/w8G/bO/wCjgvEv/kH/AON0f8PBv2zv+jgvEv8A5B/+N189UUAfQv8Aw8G/bO/6OC8S/wDkH/43R/w8G/bO/wCjgvEv/kH/AON189UUAfQv/Dwb9s7/AKOC8S/+Qf8A43R/w8G/bO/6OC8S/wDkH/43Xz1RQB9ASft+ftjStvk+P3iRj65h/wDiKK+f6KACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooq7o2i6t4h1GHSND0+e+vbhtsUECF3c+gAoApUV23i74KfFfwHpn9s+MfAOtaRZbtvn3do8aZ9MkVxNABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAV+of/BMP4Z6D4R+Eniv49+J/h3Pq+padl9LDWxkaUDp5Yxzn1Ffl5X6Z/wDBP3/goj4H+Fnw/b4YfGWX7Np+nADT5oLcNleBhhxnp3oA+2tQ8HaP+3T+zybb4l+D7zwm17M5gtpt0csLDBVyDg9cce1fFHi7/gi94h06zvLzwx8SVv3ihklhhe3VSzAEhc57nivr3VPjH8C/20vAV94I+GPxam8N6pG4+zSJL9lmD5GNqhhuHAr4R+MPhf8Abq/ZC+IlleaV4x8ReL9OCGSC6QSTxNgch05AwPWgD4t+IPwc+Jfwuvriy8b+D9S0wW87wedNbssUjKcHaxGCK4yvuex/4Ka+PfEesx6V8dPA2g6/psU2JoZtPiV0OfmH3cg17TqP7BX7OH7WGl6d47+A3jyz8N6lqxa81DSxKJWjZwDsCE/KAc9BQB+V9FfdHxn/AOCTPxy+HyteeCLmDxRZQwNNM4/duNoyQFGcmvjfxT8P/G3gmQR+LfC+paSWYqpu7dowxHXBI5oA5+iiigAooooAK7vwB8Dfiv8AE6W0HgvwPquowXkwhS6itnMAYnHLgYArE+H1v4eu/HGhWvixymjS38KX7A4IgLDefyzX7NftD/tK+DP2KvgV4Pi+Dvguyng1+x3adcbQNjBR87cHJJOeaAPyz1j9i/8AaO0XU5dLuPhrqkskWMvDA7ofocc15941+E3xI+HYV/GvgzVdIidtqS3Vs0aMfQEjmvqyT/grX+1A7s/2jRuTkZsYyf8A0Gvoj9mz9u34Z/tRY+En7UnhrRxe3gKWmoSQosZYngdAFPPWgD8nqK+uP2/P2O7T9m7xdDr/AIR1SC78K68xls1EgLxE9h6jnrXyPQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAafhzxPr3hLVbfWvDuq3Njd20gkjkhkKkMOmcda/Qn4K/8Fctf0XQf7A+NXhG38TJBCkFvMEG7A4YtxySK/OOigD9h9Q+Hv7Bv7bXhldd0LU9N8Ja9BCZ5081bcrNKOjDI3YI71y2nf8ABM/xB4V0P+0fgL8fCvia0iknmS1usrLGV/djAbjODzX5VWep6lpxY6fqFzal/vGGVkz9cGvU/hF+1N8ZPg34qh8VeGvF19JLGiRSQ3EzOkkafdUg5oA+m/hr+1f+2v8As4+LNR0Pxr4e13xHYWdyUv0ubZ5F2ITuKuQeMZr6C8P/APBRH9kr473smi/G34Z2mkRWqExTTQK3zn7w5Ax0rgPhV/wVu0jWLiXTvjh8NdMuYr5lhe4t7dGIVjhmY7R2Nej+Kf2Rf2Nv2xLe58VfBTxdZaRrUkW1bWCRQDLjuu719qAIY/g7/wAE5/2mprjwr4B16DStXVWkgbzBFlz0Aw3P0r5j+N//AASp+OXw/vzceAo4/FGlOkkwliOGRByBgA54rM+Nv/BOj9oX9nC0Xxt4cupdVtbIedLdac7B4QvOTt+lL+z/AP8ABTf45fCG8TSvFt43iPRzLGs8d1/rI41OCBx1xmgD5H8S+FPEfg/U5dH8TaPdaddxMyNFcRlDkHBxnrWVX686V+2v+wn8ftUOofFb4eWumXsGFWa4hVQ2ep7ZqWX9jr9gT9oPWZ/GHgn4hW+mwTlY/scFwFUMOOBu70AfkCOor9Yv2zPhZZeL/wBgb4feO5bto7jwzp9usadnEiAn/wBBrzP9pH/gkr4z8KmTxH8Dbw6/pTiPyrRjmbkcsMZJFfa/iL9mbxx8T/2JdI+CF1eJp2u/ZbfzDN0jKrgqc/WgD8Eq6b4aeEtc8c+OdG8L+HI7hr6+u4442gB3plh83HTFfa11/wAEivjTpHibTNLu7y2vdPvtyTXVu2RbnIAJx2r6Gfw1+zj/AMEz/AVve+IrKDxF8RLgNJbuyhnDfwkDsBkUAfFv7f3ji6n8WeHPhJLd3c7eCdMjtrqWaRj5srojZwT25FfJ9dn8YfiZqvxg+I2s/ELWY1judWnMhReir0UfliuMoAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACiiigAooooAK6HwV8QfGXw71eDXPB3iC80y7t23o0MpC5916H8a56igD7+/Zk/wCCo/jHwncXnhv4/B/FWg6n+7eRowXjDZBG0cEc19QW/gv/AIJrfHq0k0LQLvRtO1rxDGZlKXAEsMjjJ+XdjIJ6V+MNTWl7eafOt1YXc1tMn3ZIZCjD6Ec0AfpF4x/4I2+NbvX57j4aeOtKvNAfH2eW4mAc+ucZryX4jf8ABNv9rD4R3tnB4Tt7zWY7jLl9HnkKxsMfe215L4K/bc/aY+H/AIfg8MeGfiXew2FuWMaSqsrAnr8zZJr0rwN/wVE/ae8KC5XVvEia352NnnIqeXj0wKAPfv2XPGX7dPw/+OXg/wAEfFRdXi0O+iaFIL6HCGNdqn5iM5Arc/4Ke/tk/F34YfFOy+Gvw61t9EjsrZZ55olBMxcBgOfTNe8fsd/8FAvAH7TOqaZ4J8QaD9m8aQW7PveFWQ4xuKMeR+FfBf8AwVylR/2p7pVPK2EAP/fC0Aes/wDBOr9qL9qv4ofGOLStdvb3xH4addt/M8WEt+nOQPTJrif+Cx86S/HzSI47gOE04goGztOE7V98f8E2vg7pXwu/Z10i/tpY7i619ftks3lqGGcnbkcnrX5Eft3a3qOs/tN+MTfX8lytvd+XFuYkIoHQelAHz/RRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQB6/wDsp/HQ/s7/ABn0b4kPY/a7e2JguU7iFyNxHqcCv01uvif/AME4/wBrnxNP4x+Iyw2WsLCiSS6nctb7gFxgYbnGK/GulV3T7rEfQ0Afvj4t/ay/ZY+B/wAEr/Svh78RdLuI9Os3g06zsbkTSq5U7cAnJGa/Crx14r1Dxv4u1XxVqly88+o3UkxdxglSxIz+GKxGkkYYaRiPQmm0AFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAUUUUAf/Z", "reference.jpg": "..."}
for name, data in EX.items():
    open(f'/content/examples/{name}', 'wb').write(base64.b64decode(data))
print(os.listdir('/content/examples'))


In [ ]:
import torch, numpy as np
from pipeline import Pipeline
from governance import model as GM
from router import RouterEnsemble
P = Pipeline(res=512, steps=20)
router = RouterEnsemble(f"{REPO}/models/router.pt")
net = GM.load(GOV)
P.prepare_object_dirs(["object"])
print("ready | GPU GB", round(torch.cuda.memory_allocated() / 1e9, 2))


In [ ]:
import gradio as gr

ROLE_TXT = ("Rule: sketch -> shape, text -> identity, reference -> colour and material "
            "(where inputs conflict, the reference wins colour).")

def arl_generate(sketch, ref, caption, obj, ref_colour, ref_material, seed, compare):
    if sketch is None or ref is None or not caption.strip():
        raise gr.Error("Provide a sketch, a reference image and a caption.")
    attr = f"a {ref_material.strip() or 'plain'} object in {ref_colour.strip() or 'neutral'} colors"
    spec = dict(key="ui", sketch=sketch.convert("RGB"), ref=ref.convert("RGB"), caption=caption.strip(),
                obj=(obj or "").strip(), seed=int(seed), ref_object="object", attr=attr)
    it = P.make_items([spec])[0]
    gates = GM.predict_gates(net, it["x"])
    tf = (gates["g_col"], gates["g_mat"])
    rp = router.params(it)
    out = []
    if compare:
        out.append((P.run(it, shaped=False), "Fixed baseline"))
    out.append((P.run(it, p=rp, shaped=True, text_flags=tf), "ARL (router + flag-gated text)"))
    info = (
        f"{ROLE_TXT}\n\n"
        f"Governance flags (predicted conflict probability):\n"
        f"  object   {gates['g_obj']:.2f}\n  colour   {gates['g_col']:.2f}\n  material {gates['g_mat']:.2f}\n\n"
        f"Text path: colour/material words blended toward a neutral caption by the colour flag {tf[0]:.2f} and the material flag {tf[1]:.2f}.\n\n"
        f"Router settings:\n  ControlNet scale {rp['cs']:.2f}\n  sketch hold time {rp['hold']:.2f}\n"
        f"  IP-Adapter scale {rp['ip']:.2f}\n  guidance (CFG) {rp['cfg']:.2f}\n\n"
        f"Reference description used by the router: \"{attr}\"\n"
        "Note: in the pilot the ARL reached the reference colour more often than the fixed baseline "
        "but lost sketch fidelity, and no arm resolved object conflicts."
    )
    return out, info

with gr.Blocks(title="Attribute Routing Layer") as demo:
    gr.Markdown("## Attribute Routing Layer: text, sketch and reference image\n"
                "One Sheep Army, IE 643. Frozen SDXL + scribble ControlNet + IP-Adapter, driven by a trained governance network and router. "
                "Upload a sketch (white lines on black) and a reference object, write a caption that may conflict with the reference, and compare with the fixed baseline.")
    with gr.Row():
        with gr.Column():
            i_sk = gr.Image(type="pil", label="Sketch"); i_ref = gr.Image(type="pil", label="Reference image")
        with gr.Column():
            i_cap = gr.Textbox(label="Caption", value="A person holds an axe above a pile of logs, black.")
            i_obj = gr.Textbox(label="Object name (optional)", value="axe")
            i_col = gr.Textbox(label="Reference colour (for the router)", value="brown")
            i_mat = gr.Textbox(label="Reference material (for the router)", value="leather")
            i_seed = gr.Number(value=0, precision=0, label="Seed")
            i_cmp = gr.Checkbox(value=True, label="Also show the fixed baseline")
            btn = gr.Button("Generate", variant="primary")
    o_gal = gr.Gallery(label="Results", columns=2, height=480)
    o_txt = gr.Textbox(label="What the ARL decided", lines=16)
    btn.click(arl_generate, [i_sk, i_ref, i_cap, i_obj, i_col, i_mat, i_seed, i_cmp], [o_gal, o_txt])
    gr.Examples([["/content/examples/sketch.jpg", "/content/examples/reference.jpg", "A person holds an axe above a pile of logs, black.", "axe", "brown", "leather", 0, True],
                 ["/content/examples/sketch.jpg", "/content/examples/reference.jpg", "A person holds an axe above a pile of logs, made of metal.", "axe", "brown", "leather", 0, True]],
                [i_sk, i_ref, i_cap, i_obj, i_col, i_mat, i_seed, i_cmp])

demo.queue(default_concurrency_limit=1).launch(share=True, show_error=True)
